# Task 1: Load and clean support tickets
**Project:** Support Ticket Classification for VoltMart Electronics

This notebook (1) downloads a public support-ticket dataset, (2) shows the first 5 raw rows,
(3) cleans it (remove NaNs, lower-case text, drop unused columns), (4) shows the first 5 cleaned rows,
and (5) saves the result as `cleaned_tickets.csv`.

Run it from top to bottom: **Kernel -> Restart & Run All**.

In [1]:
from pathlib import Path
import io

import pandas as pd
import requests

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

DATA_URL = (
    "https://huggingface.co/datasets/interneuronai/"
    "customer_support_ticket_classification_pegasus_dataset/resolve/main/321526.csv"
)
RAW_PATH = Path("data/raw_tickets.csv")
CLEAN_PATH = Path("cleaned_tickets.csv")

## 1. Download the dataset

In [2]:
RAW_PATH.parent.mkdir(exist_ok=True)

if RAW_PATH.exists():
    print(f"Using the file already downloaded: {RAW_PATH}")
else:
    print("Downloading dataset ...")
    response = requests.get(DATA_URL, timeout=60)
    response.raise_for_status()          # stops with a clear error if the download fails
    RAW_PATH.write_bytes(response.content)
    print(f"Saved to {RAW_PATH} ({len(response.content) / 1024:.0f} KB)")

df_raw = pd.read_csv(RAW_PATH)
print("Raw shape (rows, columns):", df_raw.shape)
print("Columns:", list(df_raw.columns))

Saved to data\raw_tickets.csv (336 KB)
Raw shape (rows, columns): (3633, 2)
Columns: ['text', 'label']


## 2. Raw data: first 5 rows

In [3]:
df_raw.head()

,text,label
0,E-poçtuma daxil olmaqda çətinlik çəkirəm. Hər dəfə daxil olmaq istəyəndə 'ya...,Email
1,Veb saydım ishlemir ve men onu berpa ede bilmirem. Men serveri yeniden bashl...,Website
2,Komandamın bir üzvü üçün yeni e-poçt hesabı qurmaqda köməyə ehtiyacım var. T...,Email
3,Veb saytım çox yavaş yüklənir və bunun səbəbini bilmirəm. Bəzi performans te...,Website
4,Men coxlu spam mekdublar alıram ve onları neje dayandırajaghımı bilmirem. Me...,Email


## 3. Clean the data
Steps:
1. Tidy column names (lower-case, no spaces).
2. Drop unused columns (empty columns and leftover index columns like `Unnamed: 0`).
3. Lower-case all text.
4. Remove rows with missing values (NaN) and duplicate rows.

In [4]:
df = df_raw.copy()

# 1. tidy column names
df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]

# 2. drop unused columns
# Add any other column names you do not need to this list after looking at the raw preview above.
EXTRA_COLUMNS_TO_DROP = []
unnamed = [c for c in df.columns if c.startswith("unnamed")]
all_empty = [c for c in df.columns if df[c].isna().all()]
to_drop = sorted(set(unnamed + all_empty + EXTRA_COLUMNS_TO_DROP) & set(df.columns))
df = df.drop(columns=to_drop)
print("Dropped columns:", to_drop)

# 3. lower-case text (strip extra spaces too)
text_cols = df.select_dtypes(include="object").columns
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    df[col] = df[col].replace({"nan": pd.NA, "": pd.NA})   # astype(str) turned NaN into "nan"

# 4. remove NaNs and duplicates
before = len(df)
df = df.dropna().drop_duplicates().reset_index(drop=True)
print(f"Rows: {before} -> {len(df)} (removed {before - len(df)})")
print("Remaining NaN values:", int(df.isna().sum().sum()))

Dropped columns: []
Rows: 3633 -> 2839 (removed 794)
Remaining NaN values: 0


## 4. Cleaned data: first 5 rows

In [5]:
df.head()

,text,label
0,e-poçtuma daxil olmaqda çətinlik çəkirəm. hər dəfə daxil olmaq istəyəndə 'ya...,email
1,veb saydım ishlemir ve men onu berpa ede bilmirem. men serveri yeniden bashl...,website
2,komandamın bir üzvü üçün yeni e-poçt hesabı qurmaqda köməyə ehtiyacım var. t...,email
3,veb saytım çox yavaş yüklənir və bunun səbəbini bilmirəm. bəzi performans te...,website
4,men coxlu spam mekdublar alıram ve onları neje dayandırajaghımı bilmirem. me...,email


## 5. Save `cleaned_tickets.csv`

## 

In [6]:
df.to_csv(CLEAN_PATH, index=False)
check = pd.read_csv(CLEAN_PATH)
print(f"Saved {CLEAN_PATH} with {check.shape[0]} rows and {check.shape[1]} columns")
check.head()

Saved cleaned_tickets.csv with 2839 rows and 2 columns


,text,label
0,e-poçtuma daxil olmaqda çətinlik çəkirəm. hər dəfə daxil olmaq istəyəndə 'ya...,email
1,veb saydım ishlemir ve men onu berpa ede bilmirem. men serveri yeniden bashl...,website
2,komandamın bir üzvü üçün yeni e-poçt hesabı qurmaqda köməyə ehtiyacım var. t...,email
3,veb saytım çox yavaş yüklənir və bunun səbəbini bilmirəm. bəzi performans te...,website
4,men coxlu spam mekdublar alıram ve onları neje dayandırajaghımı bilmirem. me...,email
